# DINOv2

#### DINO

In [ ]:
# imports

import os
import torch
import numpy as np
from PIL import Image
from tqdm import tqdm
from transformers import AutoModel, AutoImageProcessor
import yaml
from collections import defaultdict
import glob

In [ ]:
'''

This cell extracts DINOv2 embeddings for each object in the Roboflow dataset (YOLOv8 format),
using the provided YOLO labels to crop objects from images. It saves the resulting embeddings,
labels, and source image names to a .npz file for later use in k-NN classification.

'''

# source: https://medium.com/data-science-in-your-pocket/getting-started-with-dinov2-installation-setup-and-inference-made-easy-be37b07d32e7
# https://huggingface.co/docs/transformers/en/model_doc/dinov2

DATASET_DIR = "./dataset" # dataset with images and labels (YOLOv8 format)
SPLIT = "train"

IMG_DIR   = os.path.join(DATASET_DIR, SPLIT, "images") # path for images
LABEL_DIR = os.path.join(DATASET_DIR, SPLIT, "labels") # path for labels

OUTPUT_FILE = f"dinov2_objects_{SPLIT}.npz" # path of output file

MODEL_NAME = "facebook/dinov2-base" # using dinov2 for object embedding extraction
DEVICE = "cuda" if torch.cuda.is_available() else "cpu" # use gpu if available


# load pretrained model dinov2
processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()

for p in model.parameters(): # no need for gradients, just inference
    p.requires_grad = False


# parse a line in the label file
def parse_label_line(line): # get class and box from a line in the label file
    parts = line.strip().split()
    if len(parts) < 5:
        return None

    class_id = int(parts[0])
    x_center, y_center, w, h = [float(x) for x in parts[1:5]]

    return class_id, (x_center, y_center, w, h)


# load labels for an image
def load_labels(image_filename):
    # get equivalent label file path for a given image file
    label_path = os.path.join(LABEL_DIR, os.path.splitext(image_filename)[0] + ".txt")

    if not os.path.exists(label_path):
        return []

    labels = []
    with open(label_path, "r") as f:
        for line in f:
            parsed = parse_label_line(line)
            if parsed:
                labels.append(parsed)

    return labels


# convert yolo format from labels, to pixels
def yolo_to_coords(box, img_w, img_h):
    # source: https://www.geeksforgeeks.org/python/python-pil-image-crop-method/
    # yolo format is (x_center, y_center, width, height) normalized to [0,1]
    # .crop() requires (left, top, right, bottom)
    x, y, w, h = box

    left = int((x - w / 2) * img_w) # * img_w to get pixel coord from normalized
    top = int((y - h / 2) * img_h)
    right = int((x + w / 2) * img_w)
    bottom = int((y + h / 2) * img_h)

    return left, top, right, bottom


# embeded crop using dinov2
def embed_crop(crop_img):
    inputs = processor(images=crop_img, return_tensors="pt")
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    with torch.no_grad():
        out = model(**inputs)

    emb = out.last_hidden_state[:, 0, :].squeeze()
    return emb.cpu().numpy()


# process Roboflow dataset
embeddings = []
labels = []
sources = []

image_files = [f for f in os.listdir(IMG_DIR) if f.lower().endswith((".jpg", ".png", ".jpeg"))]

print(f"Found {len(image_files)} images")

print("\nExtracting object embeddings\n")

# run for every image
for fname in image_files:
    img_path = os.path.join(IMG_DIR, fname)

    try:
        img = Image.open(img_path).convert("RGB")
    except:
        continue

    img_w, img_h = img.size

    yolo_labels = load_labels(fname)

    for class_id, box in yolo_labels:
        # get pixel coords from yolo format labels
        left, top, right, bottom = yolo_to_coords(box, img_w, img_h)

        x1, y1 = max(0, left), max(0, top)
        x2, y2 = min(img_w, right), min(img_h, bottom)

        if x2 <= x1 or y2 <= y1:
            continue

        # crop the image to the bounding box of the object
        crop = img.crop((x1, y1, x2, y2))

        # get embedding for the cropped object
        emb = embed_crop(crop)

        embeddings.append(emb)
        labels.append(class_id)
        sources.append(fname)


embeddings = np.stack(embeddings)

embeddings /= (np.linalg.norm(embeddings, axis=1, keepdims=True) + 1e-8) # normalization

labels = np.array(labels)
sources = np.array(sources)


# save to npz
np.savez(
    OUTPUT_FILE,
    embeddings=embeddings,
    labels=labels,
    sources=sources
)

print(f"Saved to {OUTPUT_FILE}")


## Interpretation

### kNN

In [ ]:
train = np.load("dinov2_objects_train.npz", allow_pickle=True)
test  = np.load("dinov2_objects_test.npz",  allow_pickle=True)

train_emb    = train["embeddings"]
train_labels = train["labels"]
train_sources = train["sources"]

test_emb     = test["embeddings"]
test_labels  = test["labels"]
test_sources = test["sources"]

with open("./dataset/data.yaml", "r") as f: # get class names from roboflow dataset yaml file
    CLASS_NAMES = yaml.safe_load(f)["names"]

output_lines = []

# to print and save output in a file at the end
def log(line=""):
    print(line)
    output_lines.append(line)

correct = 0

# kNN using cosine similarity
K = 5
for i in range(len(test_emb)):
    sims = train_emb @ test_emb[i] # cosine similarity between test embedding and all train embeddings
    # .argsort() gets indices of lowest to highest similarity
    # get k highest similarities (the last k)
    # reverse for the right order
    topk = sims.argsort()[-K:][::-1] # get indices of top k most similar train embeddings

    # majority vote among top k neighbors: get how many times each class appears and get the most common
    pred = np.bincount(train_labels[topk]).argmax()
    
    true_label = test_labels[i]
    is_correct = pred == true_label

    if is_correct:
        correct += 1

    log(f"[{'TRUE' if is_correct else 'FALSE'}] {CLASS_NAMES[true_label]} → predicted {CLASS_NAMES[pred]} | {test_sources[i]}")
    for rank, idx in enumerate(topk, 1):
        log(f"  {rank}. {CLASS_NAMES[train_labels[idx]]:<20} score: {sims[idx]:.3f}  ({train_sources[idx]})")
    log()


log(f"Total: {len(test_emb)}")
log(f"Correct: {correct}")
log(f"Wrong: {len(test_emb) - correct}")
log(f"Accuracy: {correct / len(test_emb) * 100:.1f}%")


with open("knn_results.txt", "w") as f:
    f.write("\n".join(output_lines))

print("\nSaved to knn_results.txt")

# (version with chunking for memory efficiency on bigger datasets)

In [ ]:
'''

This cell extracts DINOv2 embeddings for each object in the Roboflow dataset (YOLOv8 format),
using the provided YOLO labels to crop objects from images. It saves the resulting embeddings,
labels, and source image names to a .npz file for later use in k-NN classification.

'''

# source: https://medium.com/data-science-in-your-pocket/getting-started-with-dinov2-installation-setup-and-inference-made-easy-be37b07d32e7
# https://huggingface.co/docs/transformers/en/model_doc/dinov2

DATASET_DIR = "./ingredients_data" # dataset with images and labels (YOLOv8 format)
SPLIT = "test"

IMG_DIR   = os.path.join(DATASET_DIR, SPLIT, "images") # path for images
LABEL_DIR = os.path.join(DATASET_DIR, SPLIT, "labels") # path for labels

MODEL_NAME = "facebook/dinov2-base" # using dinov2 for object embedding extraction
DEVICE = "cuda" if torch.cuda.is_available() else "cpu" # use gpu if available

def main():
    chunk_id = 0

    # load pretrained model dinov2
    processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
    model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
    model.eval()

    for p in model.parameters(): # no need for gradients, just inference
        p.requires_grad = False


    # parse a line in the label file
    def parse_label_line(line): # get class and box from a line in the label file
        parts = line.strip().split()
        if len(parts) < 5:
            return None

        class_id = int(parts[0])
        x_center, y_center, w, h = [float(x) for x in parts[1:5]]

        return class_id, (x_center, y_center, w, h)


    # load labels for an image
    def load_labels(image_filename):
        # get equivalent label file path for a given image file
        label_path = os.path.join(LABEL_DIR, os.path.splitext(image_filename)[0] + ".txt")

        if not os.path.exists(label_path):
            return []

        labels = []
        with open(label_path, "r") as f:
            for line in f:
                parsed = parse_label_line(line)
                if parsed:
                    labels.append(parsed)

        return labels


    # convert yolo format from labels, to pixels
    def yolo_to_coords(box, img_w, img_h):
        # source: https://www.geeksforgeeks.org/python/python-pil-image-crop-method/
        # yolo format is (x_center, y_center, width, height) normalized to [0,1]
        # .crop() requires (left, top, right, bottom)
        x, y, w, h = box

        left = int((x - w / 2) * img_w) # * img_w to get pixel coord from normalized
        top = int((y - h / 2) * img_h)
        right = int((x + w / 2) * img_w)
        bottom = int((y + h / 2) * img_h)

        return left, top, right, bottom


    # embeded crop using dinov2
    def embed_crop(crop_img):
        inputs = processor(images=crop_img, return_tensors="pt")
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

        with torch.no_grad():
            out = model(**inputs)

        emb = out.last_hidden_state[:, 0, :].squeeze()
        return emb.cpu().numpy()


    # process Roboflow dataset
    embeddings = []
    labels = []
    sources = []

    image_files = [f for f in os.listdir(IMG_DIR) if f.lower().endswith((".jpg", ".png", ".jpeg"))]

    print(f"Found {len(image_files)} images")

    print("\nExtracting object embeddings\n")

    # run for every image
    for fname in tqdm(image_files):
        img_path = os.path.join(IMG_DIR, fname)

        try:
            with Image.open(img_path) as img:
                img = img.convert("RGB")
        except:
            continue

        img_w, img_h = img.size

        yolo_labels = load_labels(fname)

        for class_id, box in yolo_labels:
            # get pixel coords from yolo format labels
            left, top, right, bottom = yolo_to_coords(box, img_w, img_h)

            x1, y1 = max(0, left), max(0, top)
            x2, y2 = min(img_w, right), min(img_h, bottom)

            if x2 <= x1 or y2 <= y1:
                continue

            # crop the image to the bounding box of the object
            crop = img.crop((x1, y1, x2, y2))

            # get embedding for the cropped object
            emb = embed_crop(crop)

            embeddings.append(emb)
            labels.append(class_id)
            sources.append(fname)
        
        if len(embeddings) >= 1000:
            emb_array = np.stack(embeddings)
            emb_array /= (np.linalg.norm(emb_array, axis=1, keepdims=True) + 1e-8) # normalize embeddings to unit length for cosine similarity

            np.savez(f"chunk_{chunk_id}.npz",
                    embeddings=emb_array,
                    labels=np.array(labels),
                    sources=np.array(sources))

            embeddings.clear()
            labels.clear()
            sources.clear()
            chunk_id += 1
    
    if len(embeddings) > 0:
        emb_array = np.stack(embeddings)
        emb_array /= (np.linalg.norm(emb_array, axis=1, keepdims=True) + 1e-8)

        np.savez(
            f"chunk_{chunk_id}.npz",
            embeddings=emb_array,
            labels=np.array(labels),
            sources=np.array(sources)
        )
    
    print(f"Saved all chunks")


if __name__ == "__main__":
    main()

### join chunks into a single file

In [ ]:
# path to chunks
DATASET_DIR = "./ingredients_data" # dataset with images and labels (YOLOv8 format)
SPLIT = "train"

chunk_dir = f"{DATASET_DIR}/{SPLIT}_chunks"
files = sorted(glob.glob(f"{chunk_dir}/*.npz"))

all_embeddings = []
all_labels = []
all_sources = []

for f in files:
    data = np.load(f)
    all_embeddings.append(data["embeddings"])
    all_labels.append(data["labels"])
    all_sources.append(data["sources"])

embeddings = np.concatenate(all_embeddings, axis=0)
labels = np.concatenate(all_labels, axis=0)
sources = np.concatenate(all_sources, axis=0)

np.savez(
    f"dinov2_objects_{SPLIT}.npz",
    embeddings=embeddings,
    labels=labels,
    sources=sources
)

print("Merged successfully")